# Model Training Tutorial: Fine-tuning Your Genomic AI Model 🏋️

Welcome to the most exciting part of our tutorial series! In the previous two tutorials, we prepared our data and initialized our model. Now it's time for **Model Training**.

> 💡 **Learning Objectives**: Understand supervised learning principles, master different trainer usage, complete end-to-end model fine-tuning

---

## What is Supervised Fine-tuning? 🎓

**Supervised fine-tuning** is the process of adapting a pre-trained model to a specific labeled dataset. The model "learns" through the following cycle:

```
1. 📊 Make predictions on sequences in the training set
2. 🎯 Compare predictions with true labels  
3. 📉 Calculate "error" or "loss"
4. 🔧 Adjust internal weights to reduce future errors
```

This cycle is repeated for all sequences in the training data across multiple "epochs".

### 🧠 Why is Fine-tuning So Effective?

| Training Stage | Knowledge Learned | Analogy |
|---------|------------|------|
| **Pre-training** | General patterns of genomic language | 📚 Learned to "read" genomes |
| **Fine-tuning** | Task-specific specialized knowledge | 🎯 Learned to "understand" translation efficiency |

## Training Components in OmniGenBench 🔧

To start training, we need to assemble several key components:

### 1. 🗂️ **Datasets and DataLoaders**
Wrap our training, validation, and test data into PyTorch `DataLoader`s that efficiently provide batched data to the model.

### 2. 📊 **Evaluation Metrics**
Define how we measure success. For classification tasks, we'll use the **F1 score** which balances precision and recall.

### 3. ⚙️ **Optimizer**
The algorithm that updates model weights. We'll use the popular **AdamW** optimizer.

### 4. 🎯 **Trainer**
OmniGenBench provides a powerful `Trainer` class that orchestrates the entire training process.

## 🚀 OmniGenBench Trainer Selection Guide

OmniGenBench provides multiple trainer backends to meet different needs:

| Trainer Type | Base Technology | Main Advantages | Best Use Cases |
|-----------|----------|----------|------------|
| **`Trainer`** (PyTorch Native) | Pure PyTorch | 🟢 Transparent and understandable<br>🟢 Full control | 🎯 Learning and understanding<br>🎯 Single GPU training |
| **`AccelerateTrainer`** | 🤗 Accelerate | 🟡 Seamless scaling<br>🟡 Distributed-friendly | 🎯 Multi-GPU/TPU<br>🎯 Large-scale training |
| **`HFTrainer`** | 🤗 Trainer | 🔴 Feature-rich<br>🔴 Complete ecosystem | 🎯 HF users<br>🎯 Standardized workflows |

**In this tutorial, we use `AccelerateTrainer`** - it matches the complete tutorial exactly and provides excellent performance.

> 💭 **Selection Principles**: For this tutorial, we use `AccelerateTrainer` to match the complete tutorial workflow exactly.

### 🛠️ Environment Setup and Data Preparation

First, let's set up our environment and rebuild the necessary components from previous tutorials.

In [ ]:
# Install required packages
!pip install omnigenbench -U

In [1]:
import warnings
from omnigenbench import (
    ClassificationMetric,
    AccelerateTrainer,
    OmniTokenizer,
    OmniModelForSequenceClassification,
    OmniDatasetForSequenceClassification,
)

/home/yz1033/anaconda3/envs/omni/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(



    **@@ #========= @@**            ___                     _
      **@@ +----- @@**             / _ \  _ __ ___   _ __  (_)
        **@@ = @@**               | | | || '_ ` _ \ | '_ \ | |
           **@@                   | |_| || | | | | || | | || |
        @@** = **@@                \___/ |_| |_| |_||_| |_||_|
     @@** ------+ **@@
   @@** =========# **@@            ____
  @@ ---------------+ @@          / ___|  ___  _ __
 @@ ================== @@        | |  _  / _ \| '_ \
  @@ +--------------- @@         | |_| ||  __/| | | |
   @@** #========= **@@           \____| \___||_| |_|
    @@** +------ **@@
       @@** = **@@
          @@**                    ____                      _
       **@@ = @@**               | __ )   ___  _ __    ___ | |__
    **@@ -----+  @@**            |  _ \  / _ \| '_ \  / __|| '_ \
  **@@ ==========# @@**          | |_) ||  __/| | | || (__ | | | |
  @@ --------------+ @@**        |____/  \___||_| |_| \___||_| |_|



### 📊 Configure Training Parameters

Let's define all training hyperparameters. This centralized configuration matches our complete tutorial exactly.

In [2]:
# Training Configuration - matches complete tutorial exactly
model_name_or_path = "yangheng/OmniGenome-52M"
label2id = {"0": 0, "1": 1}  # 0: Low TE, 1: High TE

print("📋 Training configuration initialized!")
print(f"  🤖 Model: {model_name_or_path}")
print(f"  🏷️ Labels: {label2id}")
print(f"  🎯 Task: Translation Efficiency Prediction")

📋 Training configuration initialized!
  🤖 Model: yangheng/OmniGenome-52M
  🏷️ Labels: {'0': 0, '1': 1}
  🎯 Task: Translation Efficiency Prediction


### 🏗️ Assemble Training Components

Now, let's create all the objects needed for training, exactly as in the complete tutorial.

In [4]:
# 1. Load tokenizer - matches complete tutorial
print("🔄 Loading tokenizer...")
tokenizer = OmniTokenizer.from_pretrained(model_name_or_path)
print(f"✅ Tokenizer loaded: {model_name_or_path}")

# 2. Load datasets - matches complete tutorial exactly
print("📊 Loading datasets...")
datasets = OmniDatasetForSequenceClassification.from_huggingface(
    dataset_name="translation_efficiency_prediction",
    tokenizer=tokenizer,
    max_length=512,
    label2id=label2id
)

print(f"📊 Datasets loaded: {list(datasets.keys())}")
for split, dataset in datasets.items():
    print(f"  - {split}: {len(dataset)} samples")

🔄 Loading tokenizer...
✅ Tokenizer loaded: yangheng/OmniGenome-52M
📊 Loading datasets...
[2025-10-12 11:23:01.942] [omnigenbench 0.3.15alpha]  Dataset already downloaded and extracted at /home/yz1033/OmniGenBench/examples/translation_efficiency_prediction/__OMNIGENOME_DATA__/datasets/translation_efficiency_prediction.If you want to re-download, please delete the existing directory.
[2025-10-12 11:23:01.944] [omnigenbench 0.3.15alpha]  Detected max_length=512 in the dataset, using it as the max_length.
[2025-10-12 11:23:01.945] [omnigenbench 0.3.15alpha]  Loading data from __OMNIGENOME_DATA__/datasets/translation_efficiency_prediction/train.json...
[2025-10-12 11:23:01.959] [omnigenbench 0.3.15alpha]  Loaded 4697 examples from __OMNIGENOME_DATA__/datasets/translation_efficiency_prediction/train.json
[2025-10-12 11:23:01.961] [omnigenbench 0.3.15alpha]  Detected shuffle=True, shuffling the examples...


100%|██████████| 4697/4697 [00:04<00:00, 971.79it/s] 


[2025-10-12 11:23:06.807] [omnigenbench 0.3.15alpha]  
Label Distribution:
[2025-10-12 11:23:06.808] [omnigenbench 0.3.15alpha]  ----------------------------------------
[2025-10-12 11:23:06.810] [omnigenbench 0.3.15alpha]  Label     		Count     		Percentage
[2025-10-12 11:23:06.811] [omnigenbench 0.3.15alpha]  ----------------------------------------
[2025-10-12 11:23:06.811] [omnigenbench 0.3.15alpha]  0         		2195      		46.73%
[2025-10-12 11:23:06.812] [omnigenbench 0.3.15alpha]  1         		2502      		53.27%
[2025-10-12 11:23:06.813] [omnigenbench 0.3.15alpha]  ----------------------------------------
[2025-10-12 11:23:06.814] [omnigenbench 0.3.15alpha]  Total samples: 4697
[2025-10-12 11:23:06.856] [omnigenbench 0.3.15alpha]  Max sequence length updated -> Reset max_length=504, label_padding_length=0
[2025-10-12 11:23:06.909] [omnigenbench 0.3.15alpha]  Detected max_length=512 in the dataset, using it as the max_length.
[2025-10-12 11:23:06.912] [omnigenbench 0.3.15alpha]  L

100%|██████████| 587/587 [00:00<00:00, 984.37it/s]


[2025-10-12 11:23:07.519] [omnigenbench 0.3.15alpha]  
Label Distribution:
[2025-10-12 11:23:07.520] [omnigenbench 0.3.15alpha]  ----------------------------------------
[2025-10-12 11:23:07.521] [omnigenbench 0.3.15alpha]  Label     		Count     		Percentage
[2025-10-12 11:23:07.522] [omnigenbench 0.3.15alpha]  ----------------------------------------
[2025-10-12 11:23:07.523] [omnigenbench 0.3.15alpha]  0         		259       		44.12%
[2025-10-12 11:23:07.524] [omnigenbench 0.3.15alpha]  1         		328       		55.88%
[2025-10-12 11:23:07.524] [omnigenbench 0.3.15alpha]  ----------------------------------------
[2025-10-12 11:23:07.525] [omnigenbench 0.3.15alpha]  Total samples: 587
[2025-10-12 11:23:07.531] [omnigenbench 0.3.15alpha]  Max sequence length updated -> Reset max_length=504, label_padding_length=0
[2025-10-12 11:23:07.540] [omnigenbench 0.3.15alpha]  Detected max_length=512 in the dataset, using it as the max_length.
[2025-10-12 11:23:07.541] [omnigenbench 0.3.15alpha]  Lo

100%|██████████| 588/588 [00:00<00:00, 963.73it/s]


[2025-10-12 11:23:08.160] [omnigenbench 0.3.15alpha]  
Label Distribution:
[2025-10-12 11:23:08.161] [omnigenbench 0.3.15alpha]  ----------------------------------------
[2025-10-12 11:23:08.162] [omnigenbench 0.3.15alpha]  Label     		Count     		Percentage
[2025-10-12 11:23:08.163] [omnigenbench 0.3.15alpha]  ----------------------------------------
[2025-10-12 11:23:08.164] [omnigenbench 0.3.15alpha]  0         		258       		43.88%
[2025-10-12 11:23:08.165] [omnigenbench 0.3.15alpha]  1         		330       		56.12%
[2025-10-12 11:23:08.166] [omnigenbench 0.3.15alpha]  ----------------------------------------
[2025-10-12 11:23:08.166] [omnigenbench 0.3.15alpha]  Total samples: 588
[2025-10-12 11:23:08.173] [omnigenbench 0.3.15alpha]  Max sequence length updated -> Reset max_length=504, label_padding_length=0
📊 Datasets loaded: ['train', 'valid', 'test']
  - train: 4697 samples
  - valid: 587 samples
  - test: 588 samples


In [5]:
# 3. Initialize model - matches complete tutorial exactly
print("🤖 Initializing model...")
model = OmniModelForSequenceClassification(
    model_name_or_path,
    tokenizer,
    num_labels=2,  # Binary classification: Low TE vs High TE
)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model initialized! Parameter count: {total_params / 1e6:.1f}M")

🤖 Initializing model...


Some weights of OmniGenomeModel were not initialized from the model checkpoint at yangheng/OmniGenome-52M and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model initialized! Parameter count: 52.7M


## 🚀 Start Training with AccelerateTrainer!

Now we'll use the same training approach as the complete tutorial. The `AccelerateTrainer` will automatically handle:

- ✅ Moving data and model to the correct device (GPU or CPU)
- ✅ Iterating through training data for the specified number of epochs
- ✅ Calculating loss and updating model weights
- ✅ Evaluating the model on the validation set after each epoch
- ✅ Logging performance metrics
- ✅ Saving the best-performing model checkpoint

In [6]:
# Setup training - exactly as in complete tutorial
print("🚀 Setting up training with AccelerateTrainer...")

metric_functions = [ClassificationMetric().f1_score]

trainer = AccelerateTrainer(
    model=model,
    train_dataset=datasets["train"],
    eval_dataset=datasets["valid"],
    test_dataset=datasets["test"],
    compute_metrics=metric_functions,
)

print("🎓 Starting training...")
metrics = trainer.train()
trainer.save_model("ogb_te_finetuned")  # Matches complete tutorial

print('Metrics:', metrics)
print("\n🎉 Training completed!")

🚀 Setting up training with AccelerateTrainer...


/home/yz1033/OmniGenBench/omnigenbench/src/trainer/base_trainer.py:218: UserWarning: No optimizer provided. Defaulting to Adam optimizer with 2e-05.
  warnings.warn(


🎓 Starting training...


Evaluating: 100%|██████████| 74/74 [00:04<00:00, 18.48it/s]


[2025-10-12 11:26:45.562] [omnigenbench 0.3.15alpha]  {'f1_score': 0.029498525073746312}


...

[2025-10-12 11:29:46.582] [omnigenbench 0.3.15alpha]  {'f1_score': 0.753004005340454}


Evaluating: 100%|██████████| 74/74 [00:03<00:00, 19.62it/s]


[2025-10-12 11:31:17.962] [omnigenbench 0.3.15alpha]  {'f1_score': 0.7718309859154929}


Testing: 100%|██████████| 74/74 [00:03<00:00, 19.41it/s]


[2025-10-12 11:31:22.067] [omnigenbench 0.3.15alpha]  {'f1_score': 0.7810320781032078}
[2025-10-12 11:31:22.266] [omnigenbench 0.3.15alpha]  The path ogb_te_finetuned already exists, please set overwrite=True to overwrite it. Rename the path to ogb_te_finetuned_20251012_113122 to save it with a timestamp.
[2025-10-12 11:31:22.579] [omnigenbench 0.3.15alpha]  The model is saved to ogb_te_finetuned_20251012_113122.
Metrics: {'valid': [{'f1_score': 0.029498525073746312}, {'f1_score': 0.7174603174603175}, {'f1_score': 0.753004005340454}, {'f1_score': 0.7718309859154929}], 'best_valid': {'f1_score': 0.7718309859154929}, 'test': [{'f1_score': 0.7810320781032078}]}

🎉 Training completed!


## 🎯 Understanding Training Results

After training completes, you'll see metrics that help you understand your model's performance:

### 📊 Key Metrics to Watch:
- **F1 Score**: Balances precision and recall (higher is better)
- **Accuracy**: Overall classification accuracy
- **Loss**: How "wrong" the model's predictions are (lower is better)

### 🎯 What Good Results Look Like:
- **F1 Score > 0.7**: Good performance
- **F1 Score > 0.8**: Excellent performance
- **Stable validation metrics**: Model is learning generalizable patterns

In [7]:
# Display training summary
print("📈 Training Summary:")
print("=" * 40)
print(f"✅ Model successfully fine-tuned for translation efficiency prediction")
print(f"📊 Dataset: Rice mRNA sequences with experimental TE labels")
print(f"🎯 Task: Binary classification (High TE vs Low TE)")
print(f"💾 Model saved as: 'ogb_te_finetuned'")
print(f"🚀 Ready for inference on new sequences!")

📈 Training Summary:
✅ Model successfully fine-tuned for translation efficiency prediction
📊 Dataset: Rice mRNA sequences with experimental TE labels
🎯 Task: Binary classification (High TE vs Low TE)
💾 Model saved as: 'ogb_te_finetuned'
🚀 Ready for inference on new sequences!


## 🎯 Summary and Next Steps

🎉 Congratulations! You have successfully completed the model training tutorial. Let's review what we've accomplished:

### ✅ **Skills Mastered**

✅ **Understood supervised fine-tuning**: How pre-trained models learn specific tasks  
✅ **Mastered the AccelerateTrainer**: Professional training with minimal code  
✅ **Learned training best practices**: Proper data loading, metric selection, model saving  
✅ **Completed end-to-end training**: From raw data to trained model  
✅ **Matched complete tutorial workflow**: Consistent with the main tutorial  

**Your model is now "intelligent"!** 🧠✨

Through fine-tuning, we have transformed a general genomic foundation model into a translation efficiency prediction specialist. This trained model has been saved and is ready for making predictions on new mRNA sequences.

---

### 🚀 What's Next...

In the final tutorial, we will explore:
- 🔮 **Model inference**: Using your trained model to predict new sequences
- 📊 **Result interpretation**: Understanding and validating predictions
- 🌐 **Deployment options**: From research to production applications
- 🚀 **Real-world usage**: Applying your model to biological research

**Ready to put your trained model to work?**

> 🎊 **Milestone**: You are now a qualified genomic AI trainer!

👉 **Final Step**: Open [04_model_inference.ipynb](https://github.com/yangheng95/OmniGenBench/blob/master/examples/translation_efficiency_prediction/04_model_inference.ipynb) to complete your learning journey!